# QLoRA fine-tune Qwen2.5-3B trên Kaggle (quota riêng Colab, chạy nền 12h)
**Settings** (panel phải): Accelerator = **GPU T4 x2** (hoặc P100) · Internet = **On**.

**Chạy NỀN (khuyến nghị):** bấm **Save Version → Save & Run All (Commit)** → notebook chạy headless tới ~12h, không cần canh, không lo ngắt. Adapter lưu ở `/kaggle/working/qwen-lora` (tải về sau ở tab Output).

Self-host ≤9B, KHÔNG API ngoài. Cấu hình T4-lite ~2-3h.

In [1]:
# 1) Lấy code
%cd /kaggle/working
![ -d viettel-medreason ] || git clone https://github.com/tienndat1904/viettel-medreason.git
%cd /kaggle/working/viettel-medreason
!git pull -q

/kaggle/working
Cloning into 'viettel-medreason'...
remote: Enumerating objects: 1145, done.
remote: Counting objects: 100% (478/478), done.
remote: Compressing objects: 100% (279/279), done.
remote: Total 1145 (delta 305), reused 307 (delta 197), pack-reused 667 (from 1)
Receiving objects: 100% (1145/1145), 4.55 MiB | 21.17 MiB/s, done.
Resolving deltas: 100% (618/618), done.
/kaggle/working/viettel-medreason


In [2]:
# 2) Cài env — KHÔNG động vào numpy/transformers/torch (dùng bản Kaggle ship sẵn).
#    Chỉ update bitsandbytes/peft/accelerate.
!pip install -q -r requirements.txt
!pip install -q -U bitsandbytes peft accelerate
import numpy, torch, transformers, bitsandbytes as bnb, peft
print('numpy', numpy.__version__, '| torch', torch.__version__, '| transformers', transformers.__version__, '| bnb', bnb.__version__, '| peft', peft.__version__, '| GPU', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE')
if not numpy.__version__.startswith('2'):
    raise SystemExit('numpy != 2.x -> Restart & Run All')

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 389.2/389.2 kB 22.4 MB/s eta 0:00:00
numpy 2.0.2 | torch 2.10.0+cu128 | transformers 5.0.0 | bnb 0.49.2 | peft 0.19.1 | GPU Tesla T4


In [3]:
# 3) SMOKE-TRAIN vài bước (kiểm không OOM). CUDA_VISIBLE_DEVICES=0 -> dùng 1 GPU cho ổn định.
import os; os.environ['CUDA_VISIBLE_DEVICES'] = '0'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'   # OOM/quá chậm -> 'Qwen/Qwen2.5-1.5B-Instruct'
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl --model {MODEL} --out /kaggle/working/smoke-lora --epochs 0.05 --max-len 2048 --seed 42

config.json: 100%|█████████████████████████████| 661/661 [00:00<00:00, 3.37MB/s]
tokenizer_config.json: 7.30kB [00:00, 1.91MB/s]
vocab.json: 2.78MB [00:00, 46.1MB/s]
merges.txt: 1.67MB [00:00, 106MB/s]
tokenizer.json: 7.03MB [00:00, 125MB/s]
[data] nạp data/synthetic/train_sft.jsonl ...
[data] 1500 mẫu train
[train] bf16 (GPU hỗ trợ bf16: True)
`torch_dtype` is deprecated! Use `dtype` instead!
model.safetensors.index.json: 35.6kB [00:00, 94.8MB/s]
Fetching 2 files: 100%|███████████████████████████| 2/2 [00:20<00:00, 10.22s/it]
Download complete: 100%|████████████████████| 6.17G/6.17G [00:20<00:00, 301MB/s]
Loading weights: 100%|█| 434/434 [00:02<00:00, 185.33it/s, Materializing param=m
generation_config.json: 100%|██████████████████| 242/242 [00:00<00:00, 1.52MB/s]
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'train_runtime': '1411', 'train_samples_per_second': '0.053

In [4]:
# 4) TRAIN — T4-lite (~2-3h): 500 mẫu, 1 epoch, max_len 1536. Adapter -> /kaggle/working/qwen-lora.
#    Muốn full (chất lượng cao hơn): bỏ --max-samples, --epochs 2, --max-len 2048.
ADAPTER = '/kaggle/working/qwen-lora'
MODEL = 'Qwen/Qwen2.5-3B-Instruct'
!python scripts/train_qlora.py --data data/synthetic/train_sft.jsonl --model {MODEL} --out {ADAPTER} --max-samples 500 --epochs 1 --batch 1 --grad-accum 8 --lr 2e-4 --max-len 2048 --seed 42
!ls -la {ADAPTER}

[data] nạp data/synthetic/train_sft.jsonl ...
[data] 500 mẫu train
[train] bf16 (GPU hỗ trợ bf16: True)
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 434/434 [00:02<00:00, 201.42it/s, Materializing param=m
trainable params: 29,933,568 || all params: 3,115,872,256 || trainable%: 0.9607
warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
{'loss': '0.01771', 'grad_norm': '0.06307', 'learning_rate': '0.0001641', 'epoch': '0.32'}
{'loss': '0.00221', 'grad_norm': '0.04067', 'learning_rate': '6.715e-05', 'epoch': '0.64'}
{'loss': '0.00125', 'grad_norm': '0.0374', 'learning_rate': '2.114e-06', 'epoch': '0.96'}
{'train_runtime': '8787', 'train_samples_per_second': '0.057', 'train_steps_per_second': '0.007', 'train_loss': '0.006822', 'epoch': '1'}
100%|████████████████████████████████████████| 63/63 [2:26:27<00:00, 139.48s/it]
✅ Lưu LoRA adapter -> /kaggle/working/qwen-lora
   Điền vào configs/config.yaml: extract.lora_adapter: /kaggle

In [5]:
# 5) Trỏ config sang adapter + backend llm (inference tự bỏ few-shot khi có adapter)
import yaml
ADAPTER = '/kaggle/working/qwen-lora'
cfg = yaml.safe_load(open('configs/config.yaml', encoding='utf-8'))
cfg['extract']['backend'] = 'llm'
cfg['extract']['lora_adapter'] = ADAPTER
cfg['extract']['llm_model'] = 'Qwen/Qwen2.5-3B-Instruct'
yaml.safe_dump(cfg, open('configs/config.yaml', 'w', encoding='utf-8'), allow_unicode=True, sort_keys=False)
print('lora_adapter =', cfg['extract']['lora_adapter'])

lora_adapter = /kaggle/working/qwen-lora


In [6]:
# 6) Đo QLoRA trên 30 file dev (METRIC CHÍNH THỨC) — so với few-shot / rule
import os, shutil, glob
os.makedirs('devset/input', exist_ok=True)
for g in glob.glob('data/dev/gold/*.json'):
    n = os.path.basename(g)[:-5]
    shutil.copy(f'data/test/input/{n}.txt', f'devset/input/{n}.txt')
!python src/pipeline.py --input devset/input --output dev_pred --backend llm
!python src/eval/official_scorer.py --pred dev_pred --gold data/dev/gold

[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=129 brands=4151
[pipeline] backend=llm | 60 file | out=dev_pred
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 434/434 [00:02<00:00, 199.33it/s, Materializing param=m
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[pipeline] xong.

=== METRIC CHÍNH THỨC (n=60 file) ===
  text_score       (0.3) = 0.2855
  assertions_score (0.3) = 0.3147
  candidates_score (0.4) = 0.2579
  ---------------------------------
  FINAL            = 0.2832


In [7]:
# 7) (Khi hài lòng) chạy full 100 file + đóng gói. output.zip + qwen-lora ở /kaggle/working (tab Output).
!python src/pipeline.py --input data/test/input --output /kaggle/working/vmr_output --backend llm
!python scripts/package_submission.py --output /kaggle/working/vmr_output --input data/test/input --n 100
!cp output.zip /kaggle/working/output.zip 2>/dev/null; ls -la /kaggle/working/*.zip

[linker] backend=lexical icd_mode=lexical | icd=kb(98186) rxnorm=kb(47719) | synonyms=129 brands=4151
[pipeline] backend=llm | 100 file | out=/kaggle/working/vmr_output
`torch_dtype` is deprecated! Use `dtype` instead!
Loading weights: 100%|█| 434/434 [00:02<00:00, 199.56it/s, Materializing param=m
The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[pipeline] xong.
✅ Đã tạo output.zip (100 file hợp lệ).
-rw-r--r-- 1 root root 69420 Jul  6 11:13 /kaggle/working/output.zip
